# GTM Intelligence Engine: Growth Signal Analysis

This notebook provides a local interface to explore the high-growth intent signals generated by the Bruin pipeline.

**Source Table**: `fct_growth_signals`  
**Database**: `./data/local_data.duckdb` (Mapped from Docker)

In [1]:
import duckdb
import pandas as pd

# Set display options for better analysis
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

## 1. Connect to DuckDB

In [8]:
# Connect to the persistent DuckDB file produced by the pipeline
con = duckdb.connect('../data/local_data.duckdb', read_only=True)

# List available tables to verify the run
con.execute("SHOW TABLES").df()

,name
0,fct_growth_signals


## 2. Query High-Growth Signals

Extract the aggregated signals for our tech keywords, sorted by `signal_count` to identify high-interest repositories.

In [9]:
query = """
SELECT 
    *
FROM fct_growth_signals
ORDER BY signal_count DESC
LIMIT 20;
"""

df_top_signals = con.execute(query).df()
df_top_signals

,signal_date,company_name,repo_name,repo_url,event_type,signal_count,intent_priority,processed_at
0,2026-03-19,affaan-m,affaan-m/everything-claude-code,https://api.github.com/repos/affaan-m/everything-claude-code,WatchEvent,357,High,2026-04-04 10:31:24.906570+02:00
1,2026-03-19,jarrodwatts,jarrodwatts/claude-hud,https://api.github.com/repos/jarrodwatts/claude-hud,WatchEvent,241,High,2026-04-04 10:31:24.906570+02:00
2,2026-03-19,THU-MAIC,THU-MAIC/OpenMAIC,https://api.github.com/repos/THU-MAIC/OpenMAIC,WatchEvent,225,High,2026-04-04 10:31:24.906570+02:00
3,2026-03-19,shareAI-lab,shareAI-lab/learn-claude-code,https://api.github.com/repos/shareAI-lab/learn-claude-code,WatchEvent,181,High,2026-04-04 10:31:24.906570+02:00
4,2026-03-19,paperclipai,paperclipai/paperclip,https://api.github.com/repos/paperclipai/paperclip,WatchEvent,139,High,2026-04-04 10:31:24.906570+02:00
5,2026-03-19,unslothai,unslothai/unsloth,https://api.github.com/repos/unslothai/unsloth,WatchEvent,136,High,2026-04-04 10:31:24.906570+02:00
6,2026-03-19,openai,openai/parameter-golf,https://api.github.com/repos/openai/parameter-golf,WatchEvent,130,High,2026-04-04 10:31:24.906570+02:00
7,2026-03-19,langchain-ai,langchain-ai/open-swe,https://api.github.com/repos/langchain-ai/open-swe,WatchEvent,99,Medium,2026-04-04 10:31:24.906570+02:00
8,2026-03-19,aiming-lab,aiming-lab/AutoResearchClaw,https://api.github.com/repos/aiming-lab/AutoResearchClaw,WatchEvent,86,Medium,2026-04-04 10:31:24.906570+02:00
9,2026-03-19,langchain-ai,langchain-ai/deepagents,https://api.github.com/repos/langchain-ai/deepagents,WatchEvent,84,Medium,2026-04-04 10:31:24.906570+02:00


In [10]:
# DK - row count for debug:

count_rows = """
SELECT COUNT(*)
FROM fct_growth_signals;
"""

df_rows = con.execute(count_rows).df()
df_rows # 7851 --start-date 2026-03-19
# docker compose run --rm bruin bruin run . 
# this should be our default date to start with

,count_star()
0,7374


## 3. Analysis by Intent Priority

In [11]:
df_priority = con.execute("""
    SELECT 
        intent_priority, 
        COUNT(*) as signal_count 
    FROM fct_growth_signals 
    GROUP BY 1
""").df()

df_priority

,intent_priority,signal_count
0,Medium,149
1,Low,7218
2,High,7


## 4. Resource Cleanup

In [12]:
con.close()